# Eksplorativ ML-segmentering av supportere

## Forretningsspørsmål
Kan aktivitets- og kjøpsmønstre i et fast snapshot gi meningsfulle, dataorienterte supportersegmenter som et supplement til dagens deterministiske regler?

Dette er **unsupervised learning**: datasettet har ingen fasitlabel som modellen skal predikere. K-means leter etter struktur i de syv tillatte aktivitetsfeatureene. `rule_segment` holdes helt utenfor treningen og brukes bare til etterfølgende sammenligning.

Alle supporterdata i dette eksperimentet er syntetiske. De kan demonstrere metode og teknisk sporbarhet, men kan ikke bevise forretningsverdi eller generalisering til reelle supportere. Resultatene er derfor eksplorative og ikke produksjonsklare.

## Metodevalg og styringsrammer

Datamengden er bare 540 rader. En lokal scikit-learn-pipeline på driveren er derfor enklere å forstå, validere og spore enn en distribuert Spark MLlib-pipeline, uten at distribuert behandling gir en praktisk gevinst. Spark brukes fortsatt til innlesing, validering og tabellskriving.

Eksperimentet har tydelige guardrails:

- Ingen navn, e-post, kontaktdata, samtykke eller `marketing_allowed` skal leses inn i modellen.
- `fan_id`, snapshot-tidspunkter og `rule_segment` er ikke treningsfeatures.
- Modellen endrer aldri `marketing_allowed` og produserer ingen automatisk beslutning.
- Det opprettes ingen modellregistrering, serving endpoint eller produksjonsintegrasjon.
- Hosted MLflow i Databricks brukes uten separat tracking-server eller hardkodet identitet.

In [ ]:
import json
import tempfile

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

spark.conf.set("spark.sql.session.timeZone", "UTC")
mlflow.sklearn.autolog(disable=True)

In [ ]:
INPUT_PATH = "/Volumes/clubdata/ml/features/fan_features.parquet"
FEATURE_TABLE = "clubdata.ml.fan_features"
ASSIGNMENT_TABLE = "clubdata.ml.fan_segments_experiment"
EXPECTED_ROW_COUNT = 540
RANDOM_STATE = 42
N_INIT = 20
CANDIDATE_K = range(2, 7)
STABILITY_SEEDS = (7, 19, 43, 73, 101)

MODEL_FEATURES = [
    "recency_days",
    "matches_purchased_12m",
    "purchase_transactions_12m",
    "tickets_purchased_12m",
    "total_spend_12m",
    "cancelled_transactions_12m",
    "refunded_transactions_12m",
]
EXPECTED_SCHEMA = [
    ("fan_id", "string"),
    ("as_of_at", "timestamp"),
    ("window_start_at", "timestamp"),
    ("recency_days", "long"),
    ("matches_purchased_12m", "long"),
    ("purchase_transactions_12m", "long"),
    ("tickets_purchased_12m", "long"),
    ("total_spend_12m", "double"),
    ("cancelled_transactions_12m", "long"),
    ("refunded_transactions_12m", "long"),
    ("rule_segment", "string"),
]
NON_TRAINING_COLUMNS = {
    "fan_id",
    "as_of_at",
    "window_start_at",
    "rule_segment",
    "marketing_allowed",
}
FORBIDDEN_NAME_TOKENS = ("name", "email", "contact", "consent")
RUN_TAGS = {
    "data_classification": "synthetic_pii_free",
    "purpose": "portfolio_experiment",
    "production_ready": "false",
}

## Spark-basert datavalidering

Parquet-filen valideres før noen data flyttes til driveren. Kontrollen krever ett fast snapshot med nøyaktig forventet skjema og regresjonstall, avviser PII-/samtykkefelt ved navn og kontrollerer alle modellfeatures for null, uendelige og negative verdier.

In [ ]:
features_sdf = spark.read.parquet(INPUT_PATH)
actual_schema = [
    (field.name, field.dataType.typeName())
    for field in features_sdf.schema.fields
]
assert actual_schema == EXPECTED_SCHEMA, (
    f"Uventet skjema: {actual_schema}"
)

forbidden_columns = [
    column
    for column in features_sdf.columns
    if column == "marketing_allowed"
    or any(token in column.lower() for token in FORBIDDEN_NAME_TOKENS)
]
assert not forbidden_columns, (
    f"PII- eller samtykkefelt er ikke tillatt: {forbidden_columns}"
)

row_summary = features_sdf.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("fan_id").alias("unique_fans"),
).first()
assert row_summary["row_count"] == EXPECTED_ROW_COUNT
assert row_summary["unique_fans"] == EXPECTED_ROW_COUNT

null_counts = features_sdf.agg(
    *[
        F.sum(F.col(column).isNull().cast("long")).alias(column)
        for column in features_sdf.columns
    ]
).first().asDict()
assert sum(null_counts.values()) == 0, f"Nullverdier funnet: {null_counts}"

In [ ]:
invalid_numeric_counts = features_sdf.agg(
    *[
        F.sum(
            F.when(
                F.col(column).cast("double").isNull()
                | F.isnan(F.col(column).cast("double"))
                | (F.col(column).cast("double") == F.lit(float("inf")))
                | (F.col(column).cast("double") == F.lit(float("-inf")))
                | (F.col(column).cast("double") < 0),
                1,
            ).otherwise(0)
        ).alias(column)
        for column in MODEL_FEATURES
    ]
).first().asDict()
assert sum(invalid_numeric_counts.values()) == 0, (
    f"Ugyldige numeriske verdier: {invalid_numeric_counts}"
)

recency_summary = features_sdf.agg(
    F.min("recency_days").alias("minimum"),
    F.max("recency_days").alias("maximum"),
).first()
assert (recency_summary["minimum"], recency_summary["maximum"]) == (3, 366)

In [ ]:
validation_results = [
    ("Rader", row_summary["row_count"], EXPECTED_ROW_COUNT),
    ("Unike fans", row_summary["unique_fans"], EXPECTED_ROW_COUNT),
    ("Nullverdier", sum(null_counts.values()), 0),
    ("Ugyldige numeriske verdier", sum(invalid_numeric_counts.values()), 0),
    ("Minimum recency_days", recency_summary["minimum"], 3),
    ("Maksimum recency_days", recency_summary["maximum"], 366),
]
display(spark.createDataFrame(
    validation_results,
    "kontroll string, faktisk long, forventet long",
))

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS clubdata.ml")
(
    features_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FEATURE_TABLE)
)

## Avgrensning av treningsdata

Bare de syv numeriske aktivitetsfeatureene konverteres til pandas, etter deterministisk sortering på `fan_id`. Identifikatoren brukes kun til å bevare radrekkefølgen i Spark; den, tidsfeltene og `rule_segment` følger aldri med til pandas eller modellen. Dette er et bevisst og pragmatisk driver-valg for 540 rader.

In [ ]:
assert set(MODEL_FEATURES).isdisjoint(NON_TRAINING_COLUMNS)
assert all(
    not any(token in feature.lower() for token in FORBIDDEN_NAME_TOKENS)
    for feature in MODEL_FEATURES
)
assert "marketing_allowed" not in MODEL_FEATURES

ordered_features_sdf = features_sdf.orderBy("fan_id")
model_pdf = ordered_features_sdf.select(*MODEL_FEATURES).toPandas()
assert list(model_pdf.columns) == MODEL_FEATURES
assert model_pdf.shape == (EXPECTED_ROW_COUNT, len(MODEL_FEATURES))
assert np.isfinite(model_pdf.to_numpy(dtype=float)).all()
assert (model_pdf >= 0).all().all()

## Hosted MLflow

Gjeldende Databricks-bruker slås opp ved runtime. Eksperimentbanen bygges under brukerens eget workspace-område uten hardkodet e-post, bruker-ID, host, workspace-ID eller credentials. Kandidatene spores som nested runs under én parent run; bare den valgte pipelinen logges senere som modellartefakt.

In [ ]:
current_user = spark.sql(
    "SELECT current_user() AS current_user"
).first()["current_user"]
assert current_user

experiment_path = f"/Users/{current_user}/clubdata_fan_segmentation"
mlflow.set_experiment(experiment_path)

## Pipeline og kandidatmålinger

Hver kandidat bruker `log1p` for å dempe skjevfordelte telle- og beløpsfelt, standardisering slik at skala ikke styrer avstandene, og K-means med baseline-seed 42 og `n_init=20`. Silhouette beregnes i det transformerte rommet som K-means faktisk ser.

Stabilitet måles med Adjusted Rand Index (ARI) mellom baseline-labelene og fem faste alternative seeds. En kandidat som produserer færre faktiske klynger enn ønsket, får status ugyldig og tas ikke med i modellvalget.

In [ ]:
def build_pipeline(k, random_state):
    return Pipeline(
        steps=[
            (
                "log1p",
                FunctionTransformer(
                    np.log1p,
                    validate=False,
                    feature_names_out="one-to-one",
                ),
            ),
            ("scale", StandardScaler()),
            (
                "kmeans",
                KMeans(
                    n_clusters=k,
                    random_state=random_state,
                    n_init=N_INIT,
                ),
            ),
        ]
    )


def evaluate_candidate(k):
    pipeline = build_pipeline(k, RANDOM_STATE)
    baseline_labels = pipeline.fit_predict(model_pdf)
    _, cluster_sizes = np.unique(baseline_labels, return_counts=True)
    actual_clusters = int(len(cluster_sizes))

    result = {
        "k": int(k),
        "valid": actual_clusters == k,
        "actual_clusters": actual_clusters,
        "silhouette_score": None,
        "inertia": float(pipeline.named_steps["kmeans"].inertia_),
        "minimum_cluster_size": int(cluster_sizes.min()),
        "minimum_cluster_share": float(cluster_sizes.min() / len(model_pdf)),
        "stability_ari_mean": None,
        "stability_ari_minimum": None,
    }
    if not result["valid"]:
        return result, pipeline, baseline_labels

    transformed_features = pipeline[:-1].transform(model_pdf)
    result["silhouette_score"] = float(
        silhouette_score(transformed_features, baseline_labels)
    )

    stability_scores = []
    for seed in STABILITY_SEEDS:
        seed_labels = build_pipeline(k, seed).fit_predict(model_pdf)
        stability_scores.append(
            adjusted_rand_score(baseline_labels, seed_labels)
        )
    result["stability_ari_mean"] = float(np.mean(stability_scores))
    result["stability_ari_minimum"] = float(np.min(stability_scores))
    return result, pipeline, baseline_labels

In [ ]:
def log_run_parameters(k, random_state):
    mlflow.log_params(
        {
            "features": json.dumps(MODEL_FEATURES),
            "k": k,
            "random_state": random_state,
            "n_init": N_INIT,
            "scikit_learn_version": sklearn.__version__,
            "row_count": len(model_pdf),
        }
    )


def log_candidate_metrics(result):
    metrics = {
        "actual_clusters": result["actual_clusters"],
        "inertia": result["inertia"],
        "minimum_cluster_size": result["minimum_cluster_size"],
        "minimum_cluster_share": result["minimum_cluster_share"],
    }
    for name in (
        "silhouette_score",
        "stability_ari_mean",
        "stability_ari_minimum",
    ):
        if result[name] is not None:
            metrics[name] = result[name]
    mlflow.log_metrics(metrics)

In [ ]:
candidate_results = []
candidate_models = {}
candidate_labels = {}

with mlflow.start_run(run_name="kmeans_model_selection") as selection_run:
    mlflow.set_tags({**RUN_TAGS, "run_role": "selection_parent"})
    mlflow.log_params(
        {
            "features": json.dumps(MODEL_FEATURES),
            "candidate_k": "2,3,4,5,6",
            "random_state": RANDOM_STATE,
            "n_init": N_INIT,
            "stability_seeds": json.dumps(STABILITY_SEEDS),
            "scikit_learn_version": sklearn.__version__,
            "row_count": len(model_pdf),
        }
    )

    for k in CANDIDATE_K:
        result, pipeline, labels = evaluate_candidate(k)
        candidate_results.append(result)
        candidate_models[k] = pipeline
        candidate_labels[k] = labels

        with mlflow.start_run(run_name=f"candidate_k_{k}", nested=True):
            mlflow.set_tags(
                {
                    **RUN_TAGS,
                    "run_role": "candidate",
                    "candidate_valid": str(result["valid"]).lower(),
                }
            )
            log_run_parameters(k, RANDOM_STATE)
            log_candidate_metrics(result)

    valid_results = [result for result in candidate_results if result["valid"]]
    assert valid_results, "Ingen gyldige K-means-kandidater."
    best_result = max(
        valid_results,
        key=lambda result: (result["silhouette_score"], -result["k"]),
    )
    best_k = best_result["k"]
    mlflow.log_param("selected_k", best_k)
    mlflow.log_metric("selected_silhouette_score", best_result["silhouette_score"])
    mlflow.log_text(
        json.dumps(candidate_results, indent=2),
        "candidate_metrics.json",
    )

best_pipeline = candidate_models[best_k]
best_labels = candidate_labels[best_k]

In [ ]:
candidate_metrics_pdf = (
    pd.DataFrame(candidate_results)
    .sort_values("k", kind="stable")
    .reset_index(drop=True)
)
display(candidate_metrics_pdf)

## Deterministiske segmentnavn og profiler

K-means-labels er vilkårlige. De rangeres derfor leksikografisk fra lavere til høyere aktivitet etter medianene for:

1. kamper kjøpt, stigende
2. kjøpstransaksjoner, stigende
3. billetter, stigende
4. totalforbruk, stigende
5. recency, synkende (flere dager siden aktivitet betyr lavere aktivitet)
6. original numerisk label, stigende som siste tie-break

Kanselleringer og refusjoner vises i profilene, men brukes ikke som positiv aktivitetsrangering. Rangeringen mappes stabilt til `ML_01`, `ML_02`, osv.

In [ ]:
raw_profile_source = model_pdf.copy()
raw_profile_source["raw_cluster"] = best_labels
raw_counts = raw_profile_source.groupby("raw_cluster").size().rename("fan_count")
raw_stats = raw_profile_source.groupby("raw_cluster")[MODEL_FEATURES].agg(
    ["mean", "median"]
)
raw_stats.columns = [
    f"{feature}_{statistic}"
    for feature, statistic in raw_stats.columns
]
raw_profiles = raw_counts.to_frame().join(raw_stats).reset_index()

ranking = raw_profiles.sort_values(
    by=[
        "matches_purchased_12m_median",
        "purchase_transactions_12m_median",
        "tickets_purchased_12m_median",
        "total_spend_12m_median",
        "recency_days_median",
        "raw_cluster",
    ],
    ascending=[True, True, True, True, False, True],
    kind="stable",
)
raw_to_canonical = {
    int(raw_cluster): f"ML_{rank:02d}"
    for rank, raw_cluster in enumerate(ranking["raw_cluster"], start=1)
}
canonical_labels = np.array(
    [raw_to_canonical[int(label)] for label in best_labels],
    dtype=object,
)

profile_source = model_pdf.copy()
profile_source["ml_segment"] = canonical_labels
profile_counts = profile_source.groupby("ml_segment").size().rename("fan_count")
profile_stats = profile_source.groupby("ml_segment")[MODEL_FEATURES].agg(
    ["mean", "median"]
)
profile_stats.columns = [
    f"{feature}_{statistic}"
    for feature, statistic in profile_stats.columns
]
cluster_profiles_pdf = (
    profile_counts.to_frame()
    .join(profile_stats)
    .reset_index()
    .sort_values("ml_segment", kind="stable")
    .reset_index(drop=True)
)
assert cluster_profiles_pdf["fan_count"].sum() == EXPECTED_ROW_COUNT

## Final best-model run

Den valgte, allerede tilpassede pipelinen logges én gang som modellartefakt sammen med aggregerte, PII-frie cluster-profiler. Run-en registrerer ikke modellen i Model Registry og oppretter ingen serving endpoint.

In [ ]:
with mlflow.start_run(run_name=f"final_best_model_k_{best_k}"):
    mlflow.set_tags(
        {
            **RUN_TAGS,
            "run_role": "final_best_model",
            "selection_parent_run_id": selection_run.info.run_id,
        }
    )
    log_run_parameters(best_k, RANDOM_STATE)
    log_candidate_metrics(best_result)

    with tempfile.TemporaryDirectory() as temporary_directory:
        profile_path = f"{temporary_directory}/cluster_profiles.csv"
        cluster_profiles_pdf.to_csv(profile_path, index=False)
        mlflow.log_artifact(profile_path, artifact_path="profiles")

    mlflow.sklearn.log_model(
        best_pipeline,
        artifact_path="model",
    )

## PII-fri eksperimenttabell

Canonical labels kobles tilbake til Spark-radene med en deterministisk, nullbasert radindeks over unik `fan_id`. Ingen modellfeatures eller samtykkefelt følger med. Tabellen er et analyseartefakt med nøyaktig `fan_id`, `ml_segment` og `rule_segment`; den styrer ingen aktivering eller kontakt.

In [ ]:
identity_indexed_sdf = (
    features_sdf
    .select("fan_id", "rule_segment")
    .withColumn(
        "_row_index",
        (F.row_number().over(Window.orderBy("fan_id")) - 1).cast("long"),
    )
)
label_rows = [
    (int(row_index), str(ml_segment))
    for row_index, ml_segment in enumerate(canonical_labels)
]
labels_sdf = spark.createDataFrame(
    label_rows,
    "_row_index long, ml_segment string",
)
assignments_sdf = (
    identity_indexed_sdf
    .join(labels_sdf, on="_row_index", how="inner")
    .select("fan_id", "ml_segment", "rule_segment")
    .orderBy("fan_id")
)

assert assignments_sdf.columns == [
    "fan_id",
    "ml_segment",
    "rule_segment",
]
assert "marketing_allowed" not in assignments_sdf.columns
assignment_summary = assignments_sdf.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("fan_id").alias("unique_fans"),
    F.sum(
        (
            F.col("fan_id").isNull()
            | F.col("ml_segment").isNull()
            | F.col("rule_segment").isNull()
        ).cast("long")
    ).alias("null_rows"),
).first()
assert assignment_summary.asDict() == {
    "row_count": EXPECTED_ROW_COUNT,
    "unique_fans": EXPECTED_ROW_COUNT,
    "null_rows": 0,
}

(
    assignments_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ASSIGNMENT_TABLE)
)

## Resultater og sammenligning

Kandidatmålingene og cluster-profilene beskriver intern struktur og robusthet. Krysstabellen sammenligner eksplorative ML-segmenter med dagens deterministiske regelsegmenter.

ARI mellom disse segmenteringene er **ikke accuracy**. Målet er invariant mot labelnavn og uttrykker bare likhet mellom to partisjoneringer. `rule_segment` er en sammenligningsreferanse, ikke ground truth eller et treningsmål.

In [ ]:
display(candidate_metrics_pdf)
display(cluster_profiles_pdf)

cross_tab_sdf = (
    assignments_sdf
    .groupBy("ml_segment")
    .pivot("rule_segment")
    .count()
    .fillna(0)
    .orderBy("ml_segment")
)
display(cross_tab_sdf)

ordered_rule_segments = [
    row["rule_segment"]
    for row in (
        identity_indexed_sdf
        .orderBy("_row_index")
        .select("rule_segment")
        .collect()
    )
]
rule_comparison_ari = float(
    adjusted_rand_score(ordered_rule_segments, canonical_labels)
)
display(spark.createDataFrame(
    [("ARI mot rule_segment", rule_comparison_ari)],
    "measurement string, value double",
))

In [ ]:
guardrail_checks = [
    (
        "Ingen PII eller samtykke i trening",
        not forbidden_columns
        and set(model_pdf.columns) == set(MODEL_FEATURES),
    ),
    (
        "marketing_allowed påvirker ikke modell eller output",
        "marketing_allowed" not in model_pdf.columns
        and "marketing_allowed" not in assignments_sdf.columns,
    ),
    (
        "Assignment-output har bare tre tillatte kolonner",
        assignments_sdf.columns
        == ["fan_id", "ml_segment", "rule_segment"],
    ),
]
assert all(passed for _, passed in guardrail_checks)
display(spark.createDataFrame(
    guardrail_checks,
    "check_name string, passed boolean",
))

## Porteføljesnapshot

Cellen under samler eksperimentets aggregerte resultat i én PII-fri JSON-fil og logger den som MLflow-artefakten `portfolio_summary.json`. Artefakten kan promoteres manuelt til `data/ml/fan_segmentation_summary.json` i repoet, slik at porteføljesiden kan vise resultatet uten tilgang til Databricks eller MLflow.

Snapshotet inneholder bare aggregater: kandidatmålinger, segmentprofiler, krysstabell mot `rule_segment`, guardrail-status og beslutningen. Ingen `fan_id`, navn, e-post, kontaktdata eller samtykkefelter eksporteres.

Promoteringen er et bevisst manuelt steg og ikke en ren filkopi. Ved promotering normaliseres feltnavnene til camelCase, segmentene får lesbare etiketter og tolkninger, og `promotion`-blokken dokumenterer når og hvordan tallene ble promotert. `src/export_portfolio_data.py` validerer den promoterte filen mot `REQUIRED_ML_KEYS` og avviser den hvis segmentstørrelser, krysstabell eller guardrails ikke henger sammen.

In [ ]:
snapshot_window = features_sdf.agg(
    F.max("as_of_at").alias("as_of_at"),
    F.min("window_start_at").alias("window_start_at"),
).first()

cross_tab_pdf = cross_tab_sdf.toPandas()
rule_segment_columns = [
    column for column in cross_tab_pdf.columns if column != "ml_segment"
]
baseline_result = next(
    result for result in candidate_results if result["k"] == 2
)


def to_plain_records(frame):
    """Pandas gir numpy-typer som json.dumps ikke kan serialisere."""
    return json.loads(frame.to_json(orient="records"))


portfolio_summary = {
    "schemaVersion": 1,
    "experiment": {
        "notebook": "07_ml_fan_segmentation",
        "selectionRunId": selection_run.info.run_id,
        "snapshotAt": snapshot_window["as_of_at"].isoformat(),
        "windowStartAt": snapshot_window["window_start_at"].isoformat(),
        "rowCount": EXPECTED_ROW_COUNT,
        "randomState": RANDOM_STATE,
        "nInit": N_INIT,
        "features": list(MODEL_FEATURES),
        "stabilitySeeds": list(STABILITY_SEEDS),
    },
    "selection": {
        "selectedK": best_k,
        "baselineK": baseline_result["k"],
        "selectedSilhouette": best_result["silhouette_score"],
        "baselineSilhouette": baseline_result["silhouette_score"],
        "silhouetteDelta": (
            best_result["silhouette_score"] - baseline_result["silhouette_score"]
        ),
        "rule": (
            "Høyeste silhouette vinner, med laveste k som tie-break ved lik score."
        ),
        "candidates": [dict(result) for result in candidate_results],
    },
    "clusterProfiles": to_plain_records(cluster_profiles_pdf),
    "ruleComparison": {
        "adjustedRandIndex": rule_comparison_ari,
        "ruleSegments": rule_segment_columns,
        "crossTab": to_plain_records(cross_tab_pdf),
    },
    "guardrails": [
        {"check": check_name, "passed": bool(passed)}
        for check_name, passed in guardrail_checks
    ],
    "decision": {
        "verdict": "Eksperiment godkjent – produksjon avvist",
        "reasons": [
            "Dataene er syntetiske og beviser ikke reelle supportermønstre.",
            f"Forskjellen mellom k={best_k} og k={baseline_result['k']} er marginal.",
            "Forretningseffekten er ikke dokumentert eller faglig validert.",
        ],
        "notRegistered": [
            "modellregistrering",
            "serving endpoint",
            "automatisert supporterbeslutning",
        ],
    },
}

forbidden_snapshot_tokens = ("fan_id", "email", "consent", "contact")
serialized_summary = json.dumps(portfolio_summary, indent=2, ensure_ascii=False)
assert not any(
    token in serialized_summary.lower() for token in forbidden_snapshot_tokens
), "Porteføljesnapshotet må være fritt for identifiserende felter."
assert all(entry["passed"] for entry in portfolio_summary["guardrails"])

with mlflow.start_run(run_name="portfolio_summary_export"):
    mlflow.set_tags(
        {
            **RUN_TAGS,
            "run_role": "portfolio_summary_export",
            "selection_parent_run_id": selection_run.info.run_id,
        }
    )
    mlflow.log_text(serialized_summary, "portfolio_summary.json")

print(serialized_summary)


## Konklusjon og eksplisitte guardrails

Denne notebooken er et isolert porteføljeeksperiment på syntetiske, PII-frie data. Den demonstrerer validering, unsupervised modellseleksjon, stabilitetsmåling og sporbarhet, men beviser ikke forretningsverdi eller rettferdighet på reelle supporterdata.

Resultatet skal bare brukes til analyse. Det utløser ingen automatisk beslutning, påvirker aldri `marketing_allowed`, registrerer ingen modell, oppretter ingen serving endpoint og kobles ikke til produksjonsflyt. En eventuell videreføring krever reelle data under godkjent behandlingsgrunnlag, representativitets- og biasvurdering, tydelig eierskap, overvåking og menneskelig beslutningskontroll.